<a href="https://colab.research.google.com/github/sumairdawani/Bus-118-/blob/main/Test%20file.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part turn 1

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

# Generate data
data_file = "/content/Test file/housing_prices.csv"
df = pd.read_csv(data_file)

required_columns = {"price", "footage", "location"}
if not required_columns.issubset(df.columns):
    raise ValueError(
        "housing_prices.csv must contain price, footage, and location columns."
    )

df = df[["price", "footage", "location"]].copy()
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["footage"] = pd.to_numeric(df["footage"], errors="coerce")
df = df.dropna()

X = df[["footage", "location"]]
y = df["price"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["location"],
        )
    ],
    remainder="passthrough",
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)
model.fit(X_train, y_train)

new_house = pd.DataFrame({"footage": [2000], "location": ["Downtown"]})
predicted_price = model.predict(new_house)[0]
test_predictions = model.predict(X_test)

print(f"Records used: {len(df):,}")
print(f"Predicted price for a 2,000 sq ft house: ${predicted_price:,.2f}")
coefficients = model.named_steps["regressor"].coef_
print(f"Estimated footage coefficient: ${coefficients[-1]:,.2f} per square foot")
print(f"Test MAE: ${mean_absolute_error(y_test, test_predictions):,.2f}")
print(f"Test R²: {r2_score(y_test, test_predictions):.3f}")
print("\nModel columns: price, footage, location")
print("Source: Ames Housing dataset documented by De Cock (2011).")

Records used: 2,930
Predicted price for a 2,000 sq ft house: $233,535.60
Estimated footage coefficient: $106.69 per square foot
Test MAE: $41,382.29
Test R²: 0.523

Model columns: price, footage, location
Source: Ames Housing dataset documented by De Cock (2011).


# Part 2

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# Generate sample customer data
data = {
'age': [25, 34, 45, 28, 52, 36, 41, 29, 47, 33],
'monthly_usage_hours': [10, 50, 20, 15, 60, 30, 25, 12, 55, 40],
'purchase_amount': [100, 250, 150, 80, 300, 200, 175, 90, 280, 220],
'customer_service_calls': [5, 2, 8, 6, 1, 3, 7, 4, 0, 2],
'region': ['North', 'South', 'West', 'East', 'South', 'North', 'West', 'East',
'South', 'North'],
'churn': [1, 0, 1, 1, 0, 0, 1, 1, 0, 0] # 1 = churned, 0 = not churned
}
df = pd.DataFrame(data)
# Features and target
X = df[['age', 'monthly_usage_hours', 'purchase_amount', 'customer_service_calls',
'region']]
y = df['churn']
# Preprocessing: Scale numerical features and one-hot encode categorical features
preprocessor = ColumnTransformer(
transformers=[
('num', StandardScaler(), ['age', 'monthly_usage_hours', 'purchase_amount',
'customer_service_calls']),
('cat', OneHotEncoder(sparse_output=False), ['region'])
])
# Create pipeline with preprocessing and model
model = Pipeline(steps=[
('preprocessor', preprocessor),
('classifier', LogisticRegression(random_state=42))
])
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
random_state=42)
# Train model
model.fit(X_train, y_train)
# Predict churn probability for a new customer
new_customer = pd.DataFrame({
'age': [35],
'monthly_usage_hours': [20],
'purchase_amount': [150],
'customer_service_calls': [5],
'region': ['West']
})
churn_probability = model.predict_proba(new_customer)[0][1]  # Probability of churn (class 1)
# Classify based on threshold (0.5)
threshold = 0.5
churn_prediction = 1 if churn_probability > threshold else 0
print(f"Churn Probability for new customer: {churn_probability:.2f}")
print(f"Churn Prediction (1 = churn, 0 = no churn): {churn_prediction}")
# Display model coefficients
feature_names = [
    'age',
    'monthly_usage_hours',
    'purchase_amount',
    'customer_service_calls',
] + (
    model.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(['region'])
    .tolist()
)
coefficients = model.named_steps['classifier'].coef_[0]
print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")


Churn Probability for new customer: 0.82
Churn Prediction (1 = churn, 0 = no churn): 1

Model Coefficients:
age: -0.13
monthly_usage_hours: -0.62
purchase_amount: -0.67
customer_service_calls: 0.80
region_East: 0.17
region_North: -0.38
region_South: -0.03
region_West: 0.24


# part3

In [10]:
"""Create a clean housing-price training file from a real public dataset.

Source data:
    Ames Housing data, originally documented by Dean De Cock (2011):
    https://jse.amstat.org/v19n3/decock.pdf

    Downloaded from the public data mirror:
    https://raw.githubusercontent.com/wblakecannon/ames/master/data/housing.csv

The output keeps SalePrice and Gr Liv Area, renamed to price and footage, and
adds a reproducibly randomized location column with the three assignment
values: Downtown, Rural, and Suburb. The location labels are synthetic because
the requested random assignment is not part of the original source data.
"""
import pandas as pd
import numpy as np
from pathlib import Path


SOURCE_URL = "https://raw.githubusercontent.com/wblakecannon/ames/master/data/housing.csv"
OUTPUT_FILE = Path("/content/Test file/") / "housing_prices.csv"


# Load the real Ames Housing sales data directly from the cited source.
raw_df = pd.read_csv(SOURCE_URL)

# Keep only the target (sale price) and predictor (above-ground living area).
df = (
    raw_df[["SalePrice", "Gr Liv Area"]]
    .rename(columns={"SalePrice": "price", "Gr Liv Area": "footage"})
    .dropna()
)

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["footage"] = pd.to_numeric(df["footage"], errors="coerce")
df = df.dropna().astype({"price": "int64", "footage": "int64"})

# Add the requested location categories. The seed makes the random assignment
# reproducible whenever the dataset is regenerated.
rng = np.random.default_rng(42)
locations = ["Downtown", "Rural", "Suburb"]
df["location"] = rng.choice(locations, size=len(df))

df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(df):,} real housing-sale records to {OUTPUT_FILE.name}.")
print("Columns:", ", ".join(df.columns))
print("Source:", SOURCE_URL)

Saved 2,930 real housing-sale records to housing_prices.csv.
Columns: price, footage, location
Source: https://raw.githubusercontent.com/wblakecannon/ames/master/data/housing.csv


## Resources